# Boundary Signed Delta — Violin Plot

## What question this answers

For every algorithm-emitted boundary that was matched to a ground-truth boundary,
how far off was it — and in which direction? The **signed delta** is
`algo_frame - gt_frame`, so negative = algorithm fires early, positive = late.

## What improvement looks like

- Mean and median converge toward 0 (no systematic bias).
- The distribution gets tighter (fewer frames of error).
- Fewer outliers in the tails.
- Phantom and miss counts drop to 0.

## Red-flag patterns

- **Mean shift away from 0** → systematic bias (algorithm consistently early or late).
- **Long tails** → outlier boundaries that are very far from GT.
- **Many phantom/miss** → fundamental segment-count error, not just boundary placement.

In [ ]:
# === PARAMS ===
from pathlib import Path

SNAPSHOT_DIR = Path(r"Y:\2_Connectome\Behavior\MouseReach_Pipeline\Improvement_Snapshots\segmentation\seg_v2.2.0_multi_proposer")
FIGSIZE = (10, 6)
DPI = 300
SAVE = False  # Flip to True to write PNG + legend

In [ ]:
# === LOAD ===
import pandas as pd

deltas_path = SNAPSHOT_DIR / "metrics" / "boundary_deltas.csv"
df_all = pd.read_csv(deltas_path)

# Filter to matched rows only (phantom/miss have no meaningful signed delta)
df_matched = df_all[df_all["status"] == "matched"].copy()
df_matched["signed_delta"] = df_matched["signed_delta"].astype(int)

print(f"Loaded {len(df_all)} rows, {len(df_matched)} matched from {deltas_path.name}")

In [ ]:
# === COMPUTE ===
from mousereach.improvement.lib.palette import (
    SEGMENTATION_COLORS, SEGMENTATION_LABELS, SEGMENTATION_SUBSET_ORDER,
)

subsets = {}  # key -> {deltas, n_phantom, n_miss, label, color}

for key in SEGMENTATION_SUBSET_ORDER:
    if key == "all":
        matched = df_matched
        full = df_all
    else:
        matched = df_matched[df_matched["subset_tag"] == key]
        full = df_all[df_all["subset_tag"] == key]

    n_phantom = int((full["status"] == "phantom").sum())
    n_miss = int((full["status"] == "miss").sum())

    subsets[key] = {
        "deltas": matched["signed_delta"].values,
        "n_phantom": n_phantom,
        "n_miss": n_miss,
        "label": SEGMENTATION_LABELS[key],
        "color": SEGMENTATION_COLORS[key],
        "n_matched": len(matched),
    }

for k, v in subsets.items():
    print(f"{v['label']:30s}  n_matched={v['n_matched']:4d}  phantom={v['n_phantom']}  miss={v['n_miss']}")

In [ ]:
# === RENDER ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

positions = list(range(len(SEGMENTATION_SUBSET_ORDER)))
labels_for_y = []

for i, key in enumerate(SEGMENTATION_SUBSET_ORDER):
    s = subsets[key]
    data = s["deltas"]
    color = s["color"]
    labels_for_y.append(s["label"])

    # Violin
    if len(data) > 1:
        parts = ax.violinplot(
            data, positions=[i], vert=False, showmeans=True, showmedians=True,
            widths=0.7,
        )
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_alpha(0.5)
            pc.set_edgecolor(color)
        for partname in ("cmeans", "cmedians", "cbars", "cmins", "cmaxes"):
            if partname in parts:
                parts[partname].set_edgecolor(color)
                parts[partname].set_linewidth(1.5)

    # Jittered strip overlay
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, size=len(data))
    ax.scatter(data, i + jitter, alpha=0.25, s=8, color=color, zorder=3,
               edgecolors="none")

# Zero line
ax.axvline(0, color="#333333", linewidth=1, linestyle="--", alpha=0.6, zorder=1)

ax.set_yticks(positions)
ax.set_yticklabels(labels_for_y, fontsize=11)
ax.set_xlabel("Signed delta (frames): algo \u2212 GT", fontsize=12)
ax.set_title("Boundary signed delta (matched only)", fontsize=14, fontweight="bold")
ax.invert_yaxis()  # top-to-bottom: All, Inter-pellet, Endpoints
ax.grid(axis="x", alpha=0.3)

# Footer annotation: phantom/miss counts
footer_lines = []
for key in SEGMENTATION_SUBSET_ORDER:
    s = subsets[key]
    footer_lines.append(f"{s['label']}: N phantom: {s['n_phantom']} | N miss: {s['n_miss']}")
footer_text = "\n".join(footer_lines)
fig.text(0.12, -0.02, footer_text, fontsize=8, fontfamily="monospace",
         verticalalignment="top", color="#555555")

plt.tight_layout()
plt.show()

In [ ]:
# === SAVE ===
if SAVE:
    fig_dir = SNAPSHOT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    png_path = fig_dir / "violin.png"
    fig.savefig(str(png_path), dpi=DPI, bbox_inches="tight", pad_inches=0.15,
                facecolor="white")
    print(f"Saved: {png_path}")

    # Legend markdown
    legend_md = f"""# Boundary Signed Delta -- Violin Plot

## What question this answers

For every algorithm-emitted boundary that was matched to a ground-truth boundary,
how far off was it -- and in which direction? The **signed delta** is
`algo_frame - gt_frame`, so negative = algorithm fires early, positive = late.

## What improvement looks like

- Mean and median converge toward 0 (no systematic bias).
- The distribution gets tighter (fewer frames of error).
- Fewer outliers in the tails.
- Phantom and miss counts drop to 0.

## Red-flag patterns

- **Mean shift away from 0**: systematic bias (algorithm consistently early or late).
- **Long tails**: outlier boundaries that are very far from GT.
- **Many phantom/miss**: fundamental segment-count error, not just boundary placement.

## Rendering params

- SNAPSHOT_DIR: `{SNAPSHOT_DIR}`
- FIGSIZE: {FIGSIZE}
- DPI: {DPI}
- Subsets: {', '.join(SEGMENTATION_SUBSET_ORDER)}

## Data summary

"""
    for key in SEGMENTATION_SUBSET_ORDER:
        s = subsets[key]
        legend_md += f"- **{s['label']}**: {s['n_matched']} matched, {s['n_phantom']} phantom, {s['n_miss']} miss\n"

    legend_path = fig_dir / "violin_legend.md"
    legend_path.write_text(legend_md, encoding="utf-8")
    print(f"Saved: {legend_path}")
else:
    print("SAVE=False -- set SAVE=True to write PNG + legend")